In [1]:
import pandas as pd
import numpy as np

In [2]:
class LogisticRegression:
    def __init__(self, learning_rate, num_iterations, l2):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.weights = None
        self.bias = None
        self.l2 = l2

    def softmax(self, z):
        z -= z.max(axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def fit(self, X, y):
        X = np.array(X, dtype=np.float64)
        y = np.array(y).ravel()

        self._classes_, encoded_y = np.unique(y, return_inverse=True)
        
        num_samples, num_features = X.shape
        num_classes = len(self._classes_)
        
        self.weights = np.zeros((num_features, num_classes))
        self.bias = np.zeros((1, num_classes))
        
        y_one_hot = np.eye(num_classes)[encoded_y]

        self.loss_history = []

        for iteration in range(self.num_iterations):
            scores = np.dot(X, self.weights) + self.bias
            probabilities = self.softmax(scores)

            loss = -np.mean(np.sum(y_one_hot * np.log(probabilities + 1e-15), axis=1))
            self.loss_history.append(loss)

            error = probabilities - y_one_hot
            gradient_weights = np.dot(X.T, error) / num_samples
            gradient_bias = np.sum(error, axis=0, keepdims=True) / num_samples

            self.weights -= self.learning_rate * gradient_weights
            self.bias -= self.learning_rate * gradient_bias

        return self
    
    def predict(self, X):
        probabilities = self.predict_proba(X)
        winners = probabilities.argmax(axis=1)
        return self._classes_[winners]

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.weights.shape[0]:
            raise ValueError(f"Expected {self.weights.shape[0]} features; got {X.shape}.")
        return self.softmax(X @ self.weights + self.bias)

In [3]:
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression

train_X = pd.read_csv('data/cleaned/X_train_scaled.csv', index_col=0)
train_y = pd.read_csv('data/cleaned/y_train.csv', index_col=0)['Target']
val_X = pd.read_csv('data/cleaned/X_val_scaled.csv', index_col=0)
val_y = pd.read_csv('data/cleaned/y_val.csv', index_col=0)['Target']

test_df = pd.read_csv('data/cleaned/test_df.csv', index_col=0)

def main():
    learning_rate = 0.01
    num_iterations = 1000
    l2 = 0.01

    model_diy = LogisticRegression(learning_rate, num_iterations, l2)
    model_diy.fit(train_X, train_y)
    val_predictions = model_diy.predict(val_X)
    accuracy = np.mean(val_predictions == val_y)
    print(f'Validation Accuracy (DIY Model): {accuracy:.4f}')

    model_lib = SklearnLogisticRegression()
    model_lib.fit(train_X, train_y)
    model_lib.predict(val_X)
    accuracy = model_lib.score(val_X, val_y)
    print(f'Validation Accuracy (Library Model): {accuracy:.4f}')

if __name__ == "__main__":
    main()

Validation Accuracy (DIY Model): 0.7532
Validation Accuracy (Library Model): 0.7545


In [4]:
test_df = pd.read_csv('data/cleaned/test_df.csv', index_col=0)
student_ids = test_df['Student_ID'].to_numpy()

sample_submission = pd.read_csv('data/sample_submission.csv')
assert test_df['Student_ID'].is_unique, 'Duplicate Student_ID in test data.'
assert np.array_equal(student_ids, sample_submission['Student_ID'].to_numpy()), 'Test IDs/order differ from sample submission.'
assert train_X.columns.equals(val_X.columns), 'Training and validation features differ.'

test_X = test_df.loc[:, train_X.columns]
target_mapping = {0: 'Dropout', 1: 'Enrolled', 2: 'Graduate'}

learning_rate = 0.01
num_iterations = 1000
l2 = 0.01

def decode_targets(predictions):
    labels = pd.Series(predictions).replace(target_mapping)
    if not labels.isin(target_mapping.values()).all():
        raise ValueError('Unknown target label in predictions.')
    return labels.to_numpy()

model_lib = SklearnLogisticRegression()
model_lib.fit(train_X, train_y)

model_diy = LogisticRegression(learning_rate, num_iterations, l2)
model_diy.fit(train_X, train_y)

pred_diy = model_diy.predict(test_X.values)
pred_lib = model_lib.predict(test_X.values)

submission_diy = pd.DataFrame({'Student_ID': student_ids, 'Target': decode_targets(pred_diy)})
submission_lib = pd.DataFrame({'Student_ID': student_ids, 'Target': decode_targets(pred_lib)})

submission_diy.to_csv(f'./output/csv/submission_logres_diy.csv', index=False)
submission_lib.to_csv(f'./output/csv/submission_logres_lib.csv', index=False)


c:\Users\uhamz\miniconda3\envs\advanced_model\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [5]:
import joblib 

joblib.dump(model_diy, './output/logistic_model_diy.pkl')

['./output/logistic_model_diy.pkl']